# RSA: ColBERT + MUVERA first baselines

Choose **Runtime → Change runtime type → GPU**, then run all cells.
This compares pretrained ColBERTv2 and a MUVERA reference/Faiss HNSW baseline
on the same held-out fashion products. The binary/FP32 semantic heads use
fashion teacher supervision; ColBERT does not. Results are exploratory.

The notebook saves prepared inputs and results to Drive. Rerunning the same
configuration reuses verified embeddings. Change the run directory after a code
or configuration change. See `docs/LATE_INTERACTION.md` for metric definitions.

Subprocess output is streamed into the notebook and saved under
`MyDrive/ras_late_interaction_logs`. If a command fails, share its final
traceback or log. The wrapper exit status alone does not identify the cause.


In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

def run_logged(command, *, cwd=None, log_path):
    """Forward child stdout/stderr through notebook output and keep the failure tail."""
    import collections
    import subprocess
    import sys
    from pathlib import Path

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = collections.deque(maxlen=80)
    print(f'Python: {sys.version.split()[0]} | executable: {sys.executable}', flush=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n--- New invocation ---\n')
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                              errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if returncode:
        raise RuntimeError(
            f'Command exited with status {returncode}. Full log: {log_path}\n'
            + ''.join(tail))
    return returncode

drive.mount('/content/drive')
REPO = Path('/content/ras-late-interaction')
BRANCH = 'codex/colbert-muvera-baselines'
LOGS = Path('/content/drive/MyDrive/ras_late_interaction_logs')
if not REPO.exists():
    run_logged(['git', 'clone', '--branch', BRANCH, '--single-branch',
                'https://github.com/hanialshater/ras.git', str(REPO)],
               log_path=LOGS / 'setup.log')
else:
    run_logged(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH],
               log_path=LOGS / 'setup.log')
print(subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True))
run_logged([sys.executable, '-m', 'pip', 'install', '-e',
            str(REPO) + '[dev,benchmark,late-interaction]'],
           log_path=LOGS / 'install.log')


In [ ]:
import os
os.chdir(REPO)
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['PYTHONPATH'] = str(REPO / 'src') + ':' + str(REPO)
run_logged([sys.executable, '-m', 'pytest', '-q', 'tests/test_late_interaction.py'],
           cwd=REPO, log_path=LOGS / 'tests.log')


## Initial experiment

Start with 30 queries and one split; use 200 queries and seeds 7/17/27 after
inspecting the initial results. Reuse the same parameters when resuming.
The GPU encodes titles/images/queries; reference retrieval and MaxSim run on CPU.


In [ ]:
RUN = Path('/content/drive/MyDrive/ras_late_interaction_seed7_v1')
command = [sys.executable, '-u', '-m', 'experiments.late_interaction_baselines',
           '--output-dir', str(RUN), '--queries', '30', '--seed', '7',
           '--fde-seed', '7', '--k', '50', '--candidates', '100', '500', '1000',
           '--repetitions', '8', '--partition-bits', '4', '--fde-dim', '4096',
           '--backend', 'hnsw']
run_logged(command, cwd=REPO, log_path=LOGS / (RUN.name + '.log'))


In [ ]:
import pandas as pd
summary = pd.read_csv(RUN / 'summary.csv')
display(summary[['scope', 'method', 'candidate_budget', 'queries',
                 'recall_mean', 'precision_at_k_mean', 'ndcg_mean',
                 'fill_rate_mean', 'colbert_topk_recall_mean']])
print('Keep shared-pool recall separate from full-corpus recall.')
print('ColBERT top-K recall measures approximation, not semantic relevance.')
print('Timing and memory scope:', RUN / 'scope.json', RUN / 'memory.json')


In [ ]:
from google.colab import files
files.download(str(RUN / 'late_interaction_results.zip'))
